# 🧪 Sesión 29 · Protección Anti-CSRF y Cookies HttpOnly / SameSite

**Desarrollo de Aplicaciones Web (DAB-2201)** · TecNM Cancún  
**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`)  
**Repositorio:** https://github.com/ijmarquezl/tecnm-desarrollo-web

> **La pregunta de hoy:** ¿Por qué una cookie sola no basta para proteger una acción?

---

## 🖥️ Así se verá el resultado final

```http
POST /api/login
HTTP/1.1 200 OK
Set-Cookie: session=...; HttpOnly; SameSite=Lax

GET /api/csrf
HTTP/1.1 200 OK
{"csrf_token":"a1b2c3..."}

POST /api/logout   (sin X-CSRF-Token)
HTTP/1.1 403 Forbidden

POST /api/logout   X-CSRF-Token: a1b2c3...
HTTP/1.1 204 No Content
```

_La cookie demuestra quién eres; el token demuestra que la acción la pediste tú._

**No hay pasos intermedios.** El cuaderno declara el resultado que debe existir al final; tú decides la estructura, los modelos y el orden de tu implementación. La celda de verificación compara tu resultado con este contrato.

---

## 🎯 Al finalizar podrás

- Configurar cookies HttpOnly y SameSite.
- Implementar un token anti-CSRF.
- Verificar que una acción sensible falla sin el token.

## ✅ Criterios de aceptación

El verificador ejecutará estos casos. Cada uno debe quedar en verde:

- [ ] El login emite una cookie HttpOnly y SameSite → HTTP 200
- [ ] El logout sin token se rechaza → HTTP 403
- [ ] El token CSRF se obtiene → HTTP 200
- [ ] El logout con token funciona → HTTP 204

## 🧩 El reto

Implementa una acción sensible protegida por cookie y token anti-CSRF.

**Requisitos del contrato**

- El usuario de prueba es ana con contraseña secreta.
- POST /api/login emite una cookie HttpOnly y SameSite=Lax.
- GET /api/csrf devuelve un token para la sesión actual.
- POST /api/logout exige el token en X-CSRF-Token.
- Sin token o con token incorrecto responde 403.
- Con el token correcto responde 204.

> **Pista conceptual:** El token no se guarda en una cookie legible por JavaScript; se entrega en el cuerpo y el cliente lo reenvía en un encabezado. Así un sitio externo no puede leerlo.

## 📦 Entregable

Cuaderno en verde y bitácora `auditorias/auditoria_sesion29.md` con la diferencia entre autenticación y CSRF.


## ⚙️ Preparación del entorno

Ejecuta esta celda una sola vez. Instala las herramientas del track **Python / FastAPI** y descarga el motor de verificación del repositorio. No necesitas instalar nada en tu computadora: todo corre en Colab.


In [ ]:
import os, sys
print("⏳ Preparando el entorno de Desarrollo Web...")
!pip install -q fastapi uvicorn httpx pydantic "passlib[bcrypt]" bcrypt python-jose[cryptography] PyJWT "sqlalchemy[asyncio]" greenlet aiosqlite alembic python-multipart
!wget -q -O verificar.py https://raw.githubusercontent.com/ijmarquezl/tecnm-desarrollo-web/main/laboratorios/verificar.py
if not os.path.exists("verificar.py") or os.path.getsize("verificar.py") < 1000:
    raise RuntimeError("No se pudo descargar verificar.py. Revisa tu conexión o abre el cuaderno desde el repositorio del curso.")
print("✅ Entorno listo")
print("Python:", sys.version.split()[0])
print("Motor de verificación:", os.path.getsize("verificar.py"), "bytes")


## 🧩 El reto: escribe el resultado

Crea el o los archivos del contrato. No hay una plantilla de pasos: el esqueleto solo te da el punto de partida. Todo lo que sigue lo decides tú.

**Archivos que debes producir:** `app.py`


### 📄 `app.py`

Archivo principal de la implementación.

In [ ]:
%%writefile app.py
from fastapi import FastAPI

app = FastAPI(title="Tu protección CSRF se verá así")

# Implementa aquí el contrato declarado en el reto.
# Tú decides los modelos, las rutas, las validaciones y el manejo de errores.


## 🧪 Tu contrato C-R-E-O

Antes de pedir ayuda a la IA, escribe el contrato que describe el resultado. Esto alimenta tu bitácora y te obliga a pensar como arquitecto:

| Elemento | Tu definición |
| :--- | :--- |
| **Contexto** | Framework, versión y restricciones del entorno. |
| **Rol** | Qué papel técnico debe asumir el modelo. |
| **Especificación** | Rutas, entradas, salidas, validaciones y códigos HTTP. |
| **Output** | Qué formato exacto esperas de vuelta. |

> No pegues código de la IA sin auditarlo. La responsabilidad técnica del resultado es tuya.


## 🔍 Verificación del resultado

Ejecuta esta celda cuando tu implementación esté lista. La verificación no modifica tu código: solo observa el comportamiento y compara con el contrato. Si algo falla, el mensaje te dice **qué esperaba** y **qué recibió**.


In [ ]:
from verificar import probar_http

casos = [{'descripcion': 'El login emite una cookie HttpOnly y SameSite',
  'metodo': 'POST',
  'ruta': '/api/login',
  'json': {'usuario': 'ana', 'password': 'secreta'},
  'estado': 200,
  'headers_contiene': {'set-cookie': ['httponly', 'samesite']}},
 {'descripcion': 'El logout sin token se rechaza', 'metodo': 'POST', 'ruta': '/api/logout', 'estado': 403},
 {'descripcion': 'El token CSRF se obtiene',
  'metodo': 'GET',
  'ruta': '/api/csrf',
  'estado': 200,
  'guardar': {'csrf': 'csrf_token'}},
 {'descripcion': 'El logout con token funciona',
  'metodo': 'POST',
  'ruta': '/api/logout',
  'headers': {'X-CSRF-Token': '{csrf}'},
  'estado': 204}]

probar_http(casos, archivo="app.py", limpiar=[], env={})


## 📝 Cierre y bitácora

1. Copia la plantilla `plantillas/auditoria_sesion29.md` a `auditorias/auditoria_sesion29.md` en tu fork.
2. Responde con tus palabras:
   - ¿Qué resultado te costó más alcanzar y por qué?
   - ¿Qué decisión de diseño tomaste que no estaba escrita en el contrato?
   - ¿Qué error detectó la verificación que tú no habías previsto?
3. Haz commit de tu cuaderno ejecutado y de tu bitácora.

**La sesión termina cuando el resultado está en verde, no cuando el código "parece funcionar".**
